#### __libraries__

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [7]:
from shared_scripts.statistical_analysis import t_margin_of_error

In [10]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

#### __plt params__

In [15]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [68]:
def _2C_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index 
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance_ctrl= data.iloc[:, 0:3].mean(axis= 1),
                          avg_distance_lata= data.iloc[:, 3:6].mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance_ctrl= np.NaN,
                          avg_distance_lata= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95_ctrl= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1),
                          moe95_lata= data.apply(lambda x: t_margin_of_error(x.iloc[3:6]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95_ctrl= np.NaN,
                          moe95_lata= np.NaN)
        
    return data

#### __data load__

In [76]:
_6B= BB2_load_into_dataframe('Figure_6B.csv')

In [24]:
_6C= BB2_load_into_dataframe('Figure_6C.csv')

#### __data processing__

In [70]:
_6C_final= _2C_data_process(_6C, frequency= 6, initial_delay= 12)

In [81]:
_6B.head()

,alpha,K,group
0,0.925370,29.838429,dmso_ctrl
1,0.254590,3.459703,dmso_ctrl
2,0.023127,2.802819,dmso_ctrl
3,0.836015,19.698013,dmso_ctrl
4,0.440312,5.621942,dmso_ctrl


#### __visuals__

In [135]:
def fig6(_6C_data, **kwargs):
    
    #fig + gridspec
    fig= plt.figure(figsize= (12.5, 20), constrained_layout= True)
    gs= gridspec.GridSpec(8, 6, figure=fig)

    ax1= fig.add_subplot(gs[0:2, 0:3])
    ax2= fig.add_subplot(gs[0:2, 3:6])
    ax3= fig.add_subplot(gs[2:4, 0:6])
    ax4= fig.add_subplot(gs[4:6, 0:3])
    ax5= fig.add_subplot(gs[4:6, 3:6])
    ax6= fig.add_subplot(gs[6:7, 0:3])
    ax7= fig.add_subplot(gs[6:7, 3:6])
    ax8= fig.add_subplot(gs[7:8, 0:2])
    ax9= fig.add_subplot(gs[7:8, 2:4])
    ax10= fig.add_subplot(gs[7:8, 4:6])
    
    #fig 6C
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_ctrl, lw= 2.5, label= 'As(III)', color= '#00527C')
    y_min_ctrl= _6C_data.avg_distance_ctrl - _6C_data.moe95_ctrl
    y_max_ctrl= _6C_data.avg_distance_ctrl + _6C_data.moe95_ctrl
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#DFE9F5',  label="margin of error area: As(III)", alpha= .66)
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_lata, lw= 2.5, label= 'As(III) + LatA', color= '#FF781F')
    y_min_ctrl= _6C_data.avg_distance_lata - _6C_data.moe95_lata
    y_max_ctrl= _6C_data.avg_distance_lata + _6C_data.moe95_lata
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#FEE7CC',  label="margin of error area: As(III) + LatA", alpha= .66)
    ax3.set_xlabel('time (min)', weight= 'bold')
    ax3.set_ylabel('average movement', weight= 'bold') #pxls/frame
    ax3.set_ylim(0, 8)
    ax3.set_yticks([1, 3, 5, 7])
    ax3.legend(ncol= 2)
    ax3.set_xticks([0,60,120,180,240,300,360])

In [137]:
fig6(_6C_final)